# DSCI 522


## Lecture 7 - Environment management: containerization with Docker cont'd (and writing your own Dockerfile!)


### 2018-12-04

# Learning Objectives

## By the end of the lecture, students should be able to:

- Write a Dockerfile
- Build a Docker image from a Dockerfile on your laptop
- Build a Docker image using trusted builds on Docker Hub
- List good Docker base images for R and Python projects

## Running rocker Docker images from the Shell instead of RStudio

The default behaviour for the rocker Docker images, including the `rocker/hadleyverse` image is to launch RStudio. You can override this as follows:

```
docker run -it -e PASSWORD=yourpassword --rm rocker/tidyverse /bin/bash

```

The key is to add `/bin/bash` after the you call the image. This should also work for other images that have other default start-up behaviours.

## Installing R packages inside a Dockerfile

Technically, there are several ways to install R packages via a Dockerfile, but here is our recommended practice:

```
RUN apt-get update -qq && apt-get -y --no-install-recommends install \
  && install2.r --error \
    --deps TRUE \
    PACKAGE_NAME
```

The advantage of this strategy is that it automatically installs any R package depenedencies.



## Managing R package versions inside Docker

If your R project uses `packrat`, you can use `packrat` to manage the package versions. Another possibility is to use checkpoint/mran/snapshots from CRAN to lock-in the version:

```
RUN R -e "install.packages('gapminder', repos = 'https://mran.revolutionanalytics.com/snapshot/2016-10-19')"
```

HOWEVER - if you use this method, there can be problems with package dependency auto-installation, so use with caution. SO, what I recommend instead is versioning your Docker images (more on this later when we start building images).

## Installing Python packages inside a Dockerfile

For non-complicated packages try:

```
RUN pip3 install numpy
```

If this doesn't work, then you may have to look for examples and/or experiment.

## Managing Python package versions inside Docker

Again, I recommend versioning your Docker images... more on this later.

## Example Dockerfile for [`data_analysis_pipeline_eg`](https://github.com/ttimbers/data_analysis_pipeline_eg)

To create a `Dockerfile` that we can use to build the environment to run our data analysis pipeline in [`data_analysis_pipeline_eg`](https://github.com/ttimbers/data_analysis_pipeline_eg) we started with the `rocker/tidyverse` image and then installed the following things:

- `ezknitr` (R package)
- Python 3
- some Python dependencies (e.g., `pip3` and `python3-tk`)
- `numpy` (Python package)
- `matplotlib` (Python package)

#### Dockerfile:

```
# Docker file for data_analysis_pipeline_eg
# Tiffany Timbers, Dec, 2017

# use rocker/tidyverse as the base image and
FROM rocker/tidyverse

# then install the cowsay package
RUN apt-get update -qq && apt-get -y --no-install-recommends install \
  && install2.r --error \
    --deps TRUE \
    cowsay

# install python 3
RUN apt-get update \
  && apt-get install -y python3-pip python3-dev \
  && cd /usr/local/bin \
  && ln -s /usr/bin/python3 python \
  && pip3 install --upgrade pip

# get python package dependencies
RUN apt-get install -y python3-tk

# install numpy, pandas & matplotlib
RUN pip3 install numpy
RUN pip3 install pandas
RUN apt-get update && \
    pip3 install matplotlib && \
    rm -rf /var/lib/apt/lists/*
```

## Where does the Dockerfile live?

The Dockerfile should live in the root directory of your project.

## How do I make a Docker image from a Dockerfile?

There are 2 ways to do this! I use the first when developing my Dockerfile (to test quickly that it works), and then the second I use when I think I am "done" and want to have it archived on [Docker Hub](https://hub.docker.com/). 

1. Build a Docker image locally on your laptop

2. Build a Docker image using trusted [Automated builds](https://docs.docker.com/docker-cloud/builds/automated-build/) on [Docker Hub](https://hub.docker.com/)

### Building a Docker image locally on your laptop  
 
From the directory that contains your Dockerfile (usually your project root):

```
docker build --tag IMAGE_NAME:VERSION .
```
    
*note: `--tag` let's you name and version the Docker image. You can call this anything you want. The version number/name comes after the colon*
    
After I build, I think try to `docker run ...` to test the image locally. If I don't like it, or it doesn't work, I delete the image with `docker rmi {IMAGE_NAME}`, edit my Dockerfile and try to build and run it again.

## Running `make` interactively using Docker

If we run our `data_analysis_pipeline_eg` container and link it to our `data_analysis_pipeline_eg` directory by:

```
docker run --rm -it -e PASSWORD=yourpassword -v PATH_ON_YOUR_COMPUTER:/home/rstudio/data_analysis_eg data_analysis_pipeline_eg /bin/bash
```

Then we can `cd` into the `/home/data_analysis_eg` directory and then run `make all` (it is the `/bin/bash` that tells the container to open to bash and not the RStudio GUI).

## Running `make` non-interactively using Docker

We can do this in a more automated way. We can link the volume and drive a script or Makefile from a different directory:

```
docker run --rm -e PASSWORD=yourpassword -v PATH_ON_YOUR_COMPUTER:/home/rstudio/data_analysis_eg data_analysis_pipeline_eg make -C '/home/rstudio/data_analysis_eg'
```

The general form for this is:

```
docker run --rm -v PATH_ON_YOUR_COMPUTER:VOLUME_ON_CONTAINER DOCKER_IMAGE PROGRAM_TO_RUN PROGRAM_ARGUMENTS
```

## Building a Docker image using trusted Automated builds on Docker Hub

1. Sign-up for an account on [Docker Hub](https://hub.docker.com/)
2. Select "Create" > "Create an Automated build" and follow the instructions (you will be asked to link your Github account to your Docker Hub account).
3. Go to the Docker Hub repository you created (by following the "Create an Automated build" instructions) and click on "Build Settings", and on that page, click "Trigger"
4. After several minutes, your Docker Hub repository should say "AUTOMATED BUILD" if it successfully built and you should be able to view the Dockerfile on Docker Hub as well. 
5. To verify, delete the local image on your computer and try pulling it from Docker Hub (you will need to include your username, for example: ttimbers/data_analysis_eg).
    

## Good base images to work from for R or Python projects!

| Image | Software installed | 
|-------|--------------------|
| [rocker/tidyverse](https://hub.docker.com/r/rocker/tidyverse/) | R, R packages (including the tidyverse), RStudio, make |
| [continuumio/anaconda3](https://hub.docker.com/r/continuumio/anaconda3/) | Python 3.5, Jupyter notebook, make(?) |

*note from 2017 - currently there is a flaw in the [continuumio/anaconda3](https://hub.docker.com/r/continuumio/anaconda3/) Dockerfile and I keep getting errors with `matplotlib`... Others have documented this too... So maybe it will be more useful in the future?*

# What's next?

- conda + conda-forge as another tool for reproducibility
- reproducibility & data analysis project wrap-up